In [6]:
# ============================================================
# CHATBOT BANCOESTADO - PRUEBA 2 ING.SOLUCIONES CON IA
# Versión actualizada con:
# 1) GitHub Models API
# 2) LangChain Model API
# 3) LangChain Streaming
# 4) LangChain Memory
# 5) Prompt Engineering
# 6) RAG con Chunking + Embeddings + FAISS
# 7) Memoria conversacional indexada en RAG
# 8) Memory Summary
# ============================================================

# ----------------------------
# IMPORTS
# ----------------------------
import os
import time

from openai import OpenAI

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# ----------------------------
# 1. CONFIGURACIÓN GITHUB MODELS API
# ----------------------------
print("=== 1. Configuración GitHub Models API ===")

github_base_url = os.getenv("GITHUB_BASE_URL") or os.getenv("OPENAI_BASE_URL")
github_token = os.getenv("GITHUB_TOKEN")

if not github_base_url:
    raise ValueError("No se encontró GITHUB_BASE_URL ni OPENAI_BASE_URL.")

if not github_token:
    raise ValueError("No se encontró GITHUB_TOKEN.")

client = OpenAI(
    base_url=github_base_url,
    api_key=github_token
)

print("Base URL:", github_base_url)
print("API Key configurada:", "✓" if github_token else "✗")

os.environ["OPENAI_API_KEY"] = github_token
os.environ["OPENAI_BASE_URL"] = github_base_url

# ----------------------------
# 2. CONFIGURACIÓN LANGCHAIN MODEL API
# ----------------------------
print("\n=== 2. Configuración LangChain Model API ===")

llm = ChatOpenAI(
    base_url=github_base_url,
    api_key=github_token,
    model="gpt-4o",
    temperature=0.1,
    max_tokens=350
)

print("Modelo LangChain configurado:", llm.model_name)

# ----------------------------
# 3. CONFIGURACIÓN DE MEMORIA
# ----------------------------
print("\n=== 3. Configuración de memoria conversacional ===")

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = {
            "history": InMemoryChatMessageHistory(),
            "summary": ""
        }
    return store[session_id]

# ----------------------------
# 4. BASE DE CONOCIMIENTO BANCOESTADO
# ----------------------------
print("\n=== 4. Base de conocimiento BancoEstado ===")

documentos_bancoestado = [
    "La CuentaRUT es una cuenta vista de BancoEstado que permite realizar depósitos, transferencias, compras y giros, según las condiciones del banco.",
    "Si un cliente pierde su tarjeta, debe bloquearla inmediatamente a través de los canales oficiales de BancoEstado y luego solicitar reposición.",
    "Las transferencias pueden realizarse desde la app BancoEstado, la banca en línea o canales habilitados oficialmente por el banco.",
    "Los servicios digitales de BancoEstado permiten revisar saldo, realizar transferencias, pagar cuentas y administrar productos financieros.",
    "Nunca se deben compartir claves, PIN, coordenadas, códigos de verificación ni datos sensibles por teléfono, correo o mensajería.",
    "Para activar productos o resolver bloqueos, el cliente debe usar canales oficiales como sucursal, app, sitio web o atención telefónica oficial.",
    "La CuentaRUT puede solicitarse si el cliente cumple los requisitos definidos por BancoEstado en sus canales oficiales.",
    "Si una operación no se puede completar por medios digitales, se debe orientar al cliente a revisar la app, la web oficial o acudir a una sucursal.",
    "Las tarjetas de BancoEstado deben mantenerse protegidas y, ante uso no reconocido, se recomienda bloqueo inmediato y contacto con soporte oficial.",
    "Ante dudas sobre comisiones, requisitos, reposición o activación, el cliente debe revisar siempre la información publicada en el sitio oficial de BancoEstado."
]

print(f"Documentos cargados: {len(documentos_bancoestado)}")

# ----------------------------
# 5. CHUNKING
# ----------------------------
print("\n=== 5. Chunking de documentos ===")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    length_function=len
)

chunks = []
for doc in documentos_bancoestado:
    chunks.extend(text_splitter.split_text(doc))

print(f"Chunks generados: {len(chunks)}")

# ----------------------------
# 6. EMBEDDINGS
# ----------------------------
print("\n=== 6. Embeddings ===")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=github_token,
    base_url=github_base_url
)

# ----------------------------
# 7. VECTOR DB CONOCIMIENTO
# ----------------------------
print("\n=== 7. Base vectorial de conocimiento ===")

vector_db_banco = FAISS.from_texts(
    texts=chunks,
    embedding=embeddings
)

retriever_banco = vector_db_banco.as_retriever(search_kwargs={"k": 3})

print("Base vectorial de conocimiento creada correctamente.")

# ----------------------------
# 8. VECTOR DB MEMORIA
# ----------------------------
print("\n=== 8. Base vectorial de memoria conversacional ===")

vector_db_memoria = None

def agregar_a_memoria_vectorial(texto: str):
    global vector_db_memoria

    if vector_db_memoria is None:
        vector_db_memoria = FAISS.from_texts(
            texts=[texto],
            embedding=embeddings
        )
    else:
        vector_db_memoria.add_texts([texto])

def recuperar_memoria(pregunta: str, history, k: int = 3) -> str:
    global vector_db_memoria

    pregunta_lower = pregunta.lower().strip()

    # Consultas que requieren orden temporal exacto
    if (
        "primera pregunta" in pregunta_lower
        or "qué fue lo primero que te hice" in pregunta_lower
        or "que fue lo primero que te hice" in pregunta_lower
        or "cuál fue la primera pregunta que te hice" in pregunta_lower
        or "cual fue la primera pregunta que te hice" in pregunta_lower
    ):
        preguntas_usuario = [
            msg.content for msg in history.messages
            if msg.type == "human"
        ]

        if preguntas_usuario:
            return f"La primera pregunta registrada en esta conversación fue: {preguntas_usuario[0]}"
        else:
            return "No hay preguntas previas registradas en esta conversación."

    # Recuperación semántica normal en FAISS
    if vector_db_memoria is None:
        return "No hay memoria conversacional relevante todavía."

    docs = vector_db_memoria.similarity_search(pregunta, k=k)

    if not docs:
        return "No se encontró memoria conversacional relevante."

    return "\n".join([doc.page_content for doc in docs])

# ----------------------------
# 9. MEMORY SUMMARY
# ----------------------------
print("\n=== 9. Memory Summary ===")

def actualizar_resumen(summary_actual: str, nueva_interaccion: str) -> str:
    prompt_resumen = f"""
Resumen actual de la conversación:
{summary_actual if summary_actual else "Aún no hay resumen."}

Nueva interacción:
{nueva_interaccion}

Actualiza el resumen de la conversación de forma breve, clara y útil.
Mantén solo la información importante sobre lo que el usuario ha preguntado
y lo que el asistente ha respondido.
No inventes información.
"""

    try:
        response = llm.invoke([HumanMessage(content=prompt_resumen)])
        return response.content.strip()
    except Exception:
        # Si falla el resumen automático, conserva el anterior
        return summary_actual

# ----------------------------
# 10. PROMPT ENGINEERING
# ----------------------------
print("\n=== 10. Prompt engineering ===")

SYSTEM_PROMPT = """
Eres un asistente virtual de BancoEstado.

TAREA:
Responder preguntas sobre:
- cuentas bancarias
- tarjetas
- transferencias
- servicios digitales

OBJETIVO:
Entregar respuestas claras, breves, seguras y fáciles de entender.

FORMATO:
1. Respuesta principal
2. Pasos (si aplica)
3. Advertencia de seguridad
4. Canal oficial

RESTRICCIONES:
- No pedir datos sensibles
- No inventar información
- Distinguir entre información de la base de conocimiento, memoria conversacional y resumen conversacional
- Si no tienes información suficiente, indícalo claramente
- Derivar a canales oficiales si es necesario

REGLA IMPORTANTE:
Puedes usar:
1. contexto recuperado desde la base de conocimiento del banco
2. memoria conversacional recuperada desde interacciones previas del usuario
3. resumen conversacional acumulado

IMPORTANTE:
Los siguientes ejemplos son solo una guía de formato y estilo.
NO forman parte del historial real de la conversación.

EJEMPLO 1
Usuario: ¿Qué hago si pierdo mi tarjeta?
Asistente:
1. Respuesta principal:
Debes bloquear tu tarjeta inmediatamente.

2. Pasos:
- Ingresa a la app o web oficial
- Busca la opción de bloqueo
- Solicita reposición si corresponde

3. Advertencia de seguridad:
No compartas claves ni códigos.

4. Canal oficial:
App, sitio web oficial o atención oficial de BancoEstado.

EJEMPLO 2
Usuario: ¿Cómo creo una CuentaRUT?
Asistente:
1. Respuesta principal:
Puedes solicitar la CuentaRUT si cumples los requisitos definidos por el banco.

2. Pasos:
- Revisa los requisitos
- Solicita por canal oficial
- Sigue el proceso de activación indicado

3. Advertencia de seguridad:
Usa solo canales oficiales.

4. Canal oficial:
Sitio web, app o sucursal de BancoEstado.
"""

# ----------------------------
# 11. RECUPERACIÓN HÍBRIDA
# ----------------------------
print("\n=== 11. Recuperación híbrida RAG ===")

def recuperar_contexto_banco(pregunta: str) -> str:
    docs_relevantes = retriever_banco.invoke(pregunta)
    if not docs_relevantes:
        return "No se encontró contexto relevante en la base de conocimiento."
    return "\n".join([doc.page_content for doc in docs_relevantes])

def construir_prompt_rag(pregunta: str, history, summary: str) -> str:
    contexto_banco = recuperar_contexto_banco(pregunta)
    contexto_memoria = recuperar_memoria(pregunta, history)

    return f"""
Contexto recuperado desde la base de conocimiento BancoEstado:
{contexto_banco}

Memoria conversacional recuperada:
{contexto_memoria}

Resumen acumulado de la conversación:
{summary if summary else "Aún no hay resumen conversacional."}

Pregunta del usuario:
{pregunta}

Instrucciones:
- Responde usando primero la información más relevante entre la base de conocimiento, la memoria conversacional y el resumen.
- Si la pregunta trata sobre algo que el usuario dijo antes, prioriza la memoria conversacional y el resumen.
- Si la pregunta trata sobre productos o servicios bancarios, prioriza la base de conocimiento.
- Si no tienes información suficiente, dilo claramente.
- No inventes datos.
"""

# ----------------------------
# 12. FUNCIÓN PRINCIPAL
# ----------------------------
print("\n=== 12. Streaming + memoria + RAG habilitado ===")

def responder_con_rag_streaming_y_memoria(pregunta: str, session_id: str):
    session = get_session_history(session_id)
    history = session["history"]
    summary = session["summary"]

    prompt_contexto = construir_prompt_rag(pregunta, history, summary)

    mensajes = (
        [SystemMessage(content=SYSTEM_PROMPT)]
        + [SystemMessage(content=f"Resumen actual de la conversación:\n{summary if summary else 'Aún no hay resumen.'}")]
        + history.messages
        + [HumanMessage(content=prompt_contexto)]
    )

    respuesta_texto = ""

    for chunk in llm.stream(mensajes):
        contenido = chunk.content
        if contenido:
            print(contenido, end="", flush=True)
            respuesta_texto += contenido
            time.sleep(0.1)

    # Guardar en historial estructurado
    history.add_user_message(pregunta)
    history.add_ai_message(respuesta_texto)

    # Guardar en memoria vectorial
    agregar_a_memoria_vectorial(f"Usuario: {pregunta}")
    agregar_a_memoria_vectorial(f"Asistente: {respuesta_texto}")

    # Actualizar resumen
    nueva_interaccion = f"Usuario: {pregunta}\nAsistente: {respuesta_texto}"
    session["summary"] = actualizar_resumen(summary, nueva_interaccion)

    return respuesta_texto

# ----------------------------
# 13. CHATBOT INTERACTIVO
# ----------------------------
print("\n=== CHATBOT INTERACTIVO BANCOESTADO CON RAG ===")
print("Escribe tu consulta y presiona Enter.")
print("Para terminar, escribe: salir\n")

session_id = "bancoestado_chat"

while True:
    pregunta = input("Tú: ").strip()

    if pregunta.lower() == "salir":
        print("Chatbot: Hasta luego. Gracias por usar el asistente virtual de BancoEstado.")
        break

    if not pregunta:
        print("Chatbot: Por favor, escribe una consulta válida.")
        continue

    print("Chatbot: ", end="", flush=True)

    try:
        responder_con_rag_streaming_y_memoria(pregunta, session_id)
        print("\n")
    except Exception as e:
        print(f"\nError: {e}")
        print("Verifica tu configuración, dependencias o credenciales.\n")

# ----------------------------
# 14. HISTORIAL FINAL
# ----------------------------
print("\n=== HISTORIAL DE LA CONVERSACIÓN ===")

session = get_session_history(session_id)
historial = session["history"].messages
summary_final = session["summary"]

if historial:
    for i, msg in enumerate(historial, start=1):
        print(f"{i}. {msg.type}: {msg.content}")
else:
    print("No se registró historial.")

print("\n=== RESUMEN FINAL DE LA CONVERSACIÓN ===")
print(summary_final if summary_final else "No se generó resumen.")

=== 1. Configuración GitHub Models API ===
Base URL: https://models.inference.ai.azure.com
API Key configurada: ✓

=== 2. Configuración LangChain Model API ===
Modelo LangChain configurado: gpt-4o

=== 3. Configuración de memoria conversacional ===

=== 4. Base de conocimiento BancoEstado ===
Documentos cargados: 10

=== 5. Chunking de documentos ===
Chunks generados: 10

=== 6. Embeddings ===

=== 7. Base vectorial de conocimiento ===
Base vectorial de conocimiento creada correctamente.

=== 8. Base vectorial de memoria conversacional ===

=== 9. Memory Summary ===

=== 10. Prompt engineering ===

=== 11. Recuperación híbrida RAG ===

=== 12. Streaming + memoria + RAG habilitado ===

=== CHATBOT INTERACTIVO BANCOESTADO CON RAG ===
Escribe tu consulta y presiona Enter.
Para terminar, escribe: salir



Tú:  ¿como puedo abrir una cuenta de ahorros?


Chatbot: 1. Respuesta principal:  
Para abrir una cuenta de ahorros en BancoEstado, debes cumplir con los requisitos establecidos y realizar la solicitud a través de los canales oficiales del banco.

2. Pasos:  
- Reúne los documentos requeridos (generalmente cédula de identidad vigente y comprobante de domicilio).  
- Acércate a una sucursal de BancoEstado o revisa si puedes iniciar el proceso en la app o sitio web oficial.  
- Sigue las instrucciones del banco para completar la apertura de la cuenta.  

3. Advertencia de seguridad:  
No compartas tus datos personales ni claves en sitios o canales no oficiales.  

4. Canal oficial:  
Para más información, visita el sitio web oficial de BancoEstado, utiliza la app o acércate a una sucursal.  



Tú:  ¿que hago si me rooban la tarjeta?


Chatbot: 1. Respuesta principal:  
Si te roban la tarjeta, debes bloquearla inmediatamente para evitar usos no autorizados.

2. Pasos:  
- Ingresa a la app o sitio web oficial de BancoEstado.  
- Busca la opción de "Bloqueo de tarjeta".  
- Realiza el bloqueo y, si es necesario, solicita la reposición de la tarjeta.  
- También puedes llamar al servicio de atención al cliente o acudir a una sucursal para recibir asistencia.  

3. Advertencia de seguridad:  
No compartas tus claves ni datos personales con nadie. Usa solo los canales oficiales del banco.  

4. Canal oficial:  
App de BancoEstado, sitio web oficial o atención telefónica al 600 200 7000.



...

Chatbot: La tercera pregunta que me hiciste fue: "¿Cómo puedo abrir una CuentaRUT?".



Tú:  ¿que es una cuenta rut?


Chatbot: 1. Respuesta principal:  
La CuentaRUT es una cuenta vista de BancoEstado que permite realizar depósitos, transferencias, compras y giros, según las condiciones del banco.

2. Detalles adicionales:  
Es una cuenta básica que no tiene costos de mantención y está disponible para personas que cumplan con los requisitos establecidos por BancoEstado.

3. Advertencia de seguridad:  
Revisa siempre la información oficial sobre comisiones, requisitos y condiciones en los canales oficiales del banco.

4. Canal oficial:  
Para más información, visita el sitio web oficial de BancoEstado o utiliza la app del banco.



Tú:  salir


Chatbot: Hasta luego. Gracias por usar el asistente virtual de BancoEstado.

=== HISTORIAL DE LA CONVERSACIÓN ===
1. human: ¿como puedo abrir una cuenta de ahorros?
2. ai: 1. Respuesta principal:  
Para abrir una cuenta de ahorros en BancoEstado, debes cumplir con los requisitos establecidos y realizar la solicitud a través de los canales oficiales del banco.

2. Pasos:  
- Reúne los documentos requeridos (generalmente cédula de identidad vigente y comprobante de domicilio).  
- Acércate a una sucursal de BancoEstado o revisa si puedes iniciar el proceso en la app o sitio web oficial.  
- Sigue las instrucciones del banco para completar la apertura de la cuenta.  

3. Advertencia de seguridad:  
No compartas tus datos personales ni claves en sitios o canales no oficiales.  

4. Canal oficial:  
Para más información, visita el sitio web oficial de BancoEstado, utiliza la app o acércate a una sucursal.  
3. human: ¿que hago si me rooban la tarjeta?
4. ai: 1. Respuesta principal:  
Si te 